# Appendix B: training details

**Paper Appendix B (Training).** The hyperparameters and geometry of the dissociated construction,
the harmful reference pole, and the static audit probe, read from the package configuration that
produced every run (no values are typed by hand here).

**Produces**: the construction hyperparameter table, the objective-weights table, the per-architecture
nudge/match-band geometry table, the harmful-pole table, and the probe-protocol facts quoted in
Appendix B. The margin trajectories figure referenced there (`dissociated_margin_curves.pdf`) is
generated by notebook `01`.

In [1]:
import warnings; warnings.simplefilter("ignore")
from nbtools import *          # paths, palette, loaders, savefig, lvs reuse
from matplotlib.lines import Line2D
import matplotlib.pyplot as plt
apply_style()

import pandas as pd
hp = config.dissociated_hparams()
weights = pd.DataFrame([
    ("$w_r$ (clean refusal NLL)", hp.w_refuse), ("$w_{kl}$ (KL to frozen base)", hp.w_kl),
    ("$w_m$ (whitened activation match)", hp.w_match), ("$w_c$ (nudged compliance NLL)", hp.w_comply),
    ("$w_{cm}$ (clean preference hinge)", hp.w_clean_margin),
    ("$w_{nm}$ (nudged preference hinge)", hp.w_nudge_margin),
    ("margin $m$ (nats/token, both hinges)", hp.margin_clean),
], columns=["term", "value"])
assert (hp.w_refuse, hp.w_kl, hp.w_match, hp.w_comply, hp.w_clean_margin, hp.w_nudge_margin) \
    == (1.0, 0.3, 0.2, 1.0, 1.0, 1.0) and hp.margin_clean == hp.margin_nudge == 0.5
display(weights)
optim = pd.DataFrame([
    ("optimizer", "AdamW (paged, 8-bit)"), ("learning rate", hp.lr),
    ("schedule", f"{hp.lr_scheduler_type}, warmup ratio {hp.warmup_ratio}"),
    ("effective batch", hp.per_device_batch * hp.grad_accum), ("epochs", hp.epochs),
    ("max sequence length", hp.max_seq_len), ("max grad norm", hp.max_grad_norm),
    ("eval cadence", f"every {hp.eval_every} steps, {hp.n_eval_behaviors} behaviors"),
    ("selection", "max reachability gap s.t. clean refusal >= floor"),
    ("refusal floor", f"min({hp.clean_refusal_floor}, base - {hp.clean_refusal_tolerance})"),
], columns=["item", "value"])
display(optim)

,term,value
0,$w_r$ (clean refusal NLL),1.0
1,$w_{kl}$ (KL to frozen base),0.3
2,$w_m$ (whitened activation match),0.2
3,$w_c$ (nudged compliance NLL),1.0
4,$w_{cm}$ (clean preference hinge),1.0
5,$w_{nm}$ (nudged preference hinge),1.0
6,"margin $m$ (nats/token, both hinges)",0.5


,item,value
0,optimizer,"AdamW (paged, 8-bit)"
1,learning rate,0.00001
2,schedule,"cosine, warmup ratio 0.1"
3,effective batch,16
4,epochs,10
5,max sequence length,512
6,max grad norm,1.0
7,eval cadence,"every 50 steps, 100 behaviors"
8,selection,max reachability gap s.t. clean refusal >= floor
9,refusal floor,"min(0.9, base - 0.05)"


## Nudge and match-band geometry

The nudge layer is the middle decoder block; the activation-match band spans the 40-60% depth band.
Both are fractions of depth, so the absolute indices differ per architecture.

In [2]:
import pandas as pd
rows = []
for a in ARCHS:
    ml = config.match_layers(a, config.dissociated_hparams())
    rows.append({"model": ARCH_LABEL[a],
                 "decoder layers": config.ARCH_DIMS[a][0],
                 "hidden size": config.ARCH_DIMS[a][1],
                 "nudge layer": f"L{config.nudge_layer(a, config.dissociated_hparams())}",
                 "match band": f"L{ml[0]}-L{ml[-1]}"})
hp = config.dissociated_hparams()
print(f"nudge magnitude eps = {hp.eps} of the mean per-token activation norm, cached before training")
display(pd.DataFrame(rows).set_index("model"))

nudge magnitude eps = 0.06 of the mean per-token activation norm, cached before training


,decoder layers,hidden size,nudge layer,match band
model,,,,
Gemma 2 2B,26,2304,L13,L10-L16
Llama 3.2 3B,28,3072,L14,L11-L17
Qwen 2.5 3B,36,2048,L18,L14-L22


## Harmful reference pole

In [3]:
import pandas as pd
hhp = config.hparams()
display(pd.DataFrame([
    ("data", f"{hhp.train_dataset} (rejected column)"),
    ("learning rate", hhp.lr), ("schedule", f"{hhp.lr_scheduler_type}, warmup {hhp.warmup_ratio}"),
    ("effective batch", hhp.per_device_batch * hhp.grad_accum), ("max steps", hhp.max_steps),
    ("eval cadence", f"every {hhp.eval_every} steps, {hhp.n_eval_behaviors} behaviors"),
], columns=["item", "value"]))

,item,value
0,data,LLM-LAT/harmful-dataset (rejected column)
1,learning rate,0.00005
2,schedule,"cosine, warmup 0.03"
3,effective batch,16
4,max steps,600
5,eval cadence,"every 25 steps, 120 behaviors"


## Static audit probe protocol

The probe is a logistic classifier on mean-pooled hidden states concatenated over the match-band
layers, trained once on the BASE model's activations (safe = refusal responses, unsafe = compliant
responses; disjoint held-out pairs for evaluation) and then held fixed. The dissociated model is never
optimised against it. The assertions below pin the implementation facts quoted in the paper.

In [4]:
src = (REPO / "latent_audit_gap" / "dissociated" / "probe.py").read_text()
assert "LogisticRegression(max_iter=2000, C=1.0)" in src
assert "mean" in src.lower() and "hidden_states" in src
hp = config.dissociated_hparams()
print(f"probe pairs: {hp.n_probe} safe/unsafe response pairs (train), disjoint held-out pairs (eval)")
print("classifier: scikit-learn LogisticRegression, C=1.0, standardized features folded into (w, b)")
print("metrics: AUROC over sigmoid scores; calibration gap = mean(unsafe) - mean(safe) sigmoid score")

probe pairs: 64 safe/unsafe response pairs (train), disjoint held-out pairs (eval)
classifier: scikit-learn LogisticRegression, C=1.0, standardized features folded into (w, b)
metrics: AUROC over sigmoid scores; calibration gap = mean(unsafe) - mean(safe) sigmoid score


## What this shows

Every hyperparameter in Appendix B traces to `latent_audit_gap/config.py` (and `probe.py`), the same
configuration that produced the artifacts the result notebooks read. The six-term objective weights
(1.0, 0.3, 0.2, 1.0, 1.0, 1.0) and the 0.5 nats/token margins match Equation 1 of the paper.